# 🚀 Notebook do Professor (Demo) — Aula 13: LangGraph — StateGraph, nodes, conditional edges e HITL

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 13/14 — Módulo 4 · Grafos de estado em código**  
**1h40min**  
**StateGraph · MemorySaver · HITL · draw_mermaid**  
**Andaime 60%**  

---

## Como usar este notebook

- Cada célula corresponde a um slide de código da aula (a ordem é a da apresentação).
- Rode ao vivo enquanto explica o slide correspondente.
- A última seção traz o lab do aluno com o gabarito das lacunas.

---

# 🔬 Código da aula — slide a slide

### Slide 07 — StateGraph — estrutura minima funcional

In [ ]:
!pip install langchain-ollama langchain-core -q

from langchain_ollama import ChatOllama
from google.colab import userdata
import os

# Definir a API key via variável de ambiente (Colab Secrets)
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
!pip install langgraph langchain-ollama -q

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")
llm = ChatOllama(model="gpt-oss:120b", temperature=0)

# 1. Definir o Estado com TypedDict
class Estado(TypedDict):
    # Annotated[list, add_messages] usa reducer que ACUMULA (nao substitui)
    mensagens: Annotated[list, add_messages]
    iteracoes: int   # campo simples — substituido a cada update
    rota: str         # campo simples — a decisao do classificador

# 2. Definir Nodes — funcoes que recebem state e retornam dict
def node_responder(state: Estado) -> dict:
    resposta = llm.invoke(state["mensagens"])
    return {
        "mensagens": [resposta],  # add_messages ACUMULA — nao substitui
        "iteracoes": state["iteracoes"] + 1,
    }

# 3. Construir o Grafo
builder = StateGraph(Estado)
builder.add_node("responder", node_responder)
builder.add_edge(START, "responder")
builder.add_edge("responder", END)

# 4. Compilar — valida e retorna Runnable
app = builder.compile()

# 5. Invocar como qualquer chain LCEL
resultado = app.invoke({"mensagens": [HumanMessage("Oi!")], "iteracoes": 0, "rota": ""})
print(resultado["mensagens"][-1].content)

### Slide 09 — add_conditional_edges() — o superpoder do LangGraph

In [ ]:
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel
from typing import TypedDict, Annotated, Literal
from langgraph.graph.message import add_messages

class Estado(TypedDict):
    mensagens: Annotated[list, add_messages]
    rota: str
    iteracoes: int

class Rota(BaseModel):
    destino: Literal["rag", "calculadora", "conversa"]

# Node: classifica a intencao e escreve "rota" no estado
def node_classificar(state: Estado) -> dict:
    ultima_msg = state["mensagens"][-1].content
    rota_obj   = chain_clf.invoke({"input": ultima_msg})
    return {"rota": rota_obj.destino}

# Funcao de roteamento — le o estado e retorna string
def decidir_rota(state: Estado) -> str:
    return state["rota"]  # deve retornar uma das chaves do mapa abaixo

builder = StateGraph(Estado)
builder.add_node("classificar", node_classificar)
builder.add_node("rag",         node_rag)
builder.add_node("calculadora", node_calc)
builder.add_node("conversa",    node_chat)
builder.add_edge(START, "classificar")

# add_conditional_edges(no_origem, fn_rota, {retorno_fn: nome_no_destino})
builder.add_conditional_edges(
    "classificar",
    decidir_rota,
    {"rag":"rag", "calculadora":"calculadora", "conversa":"conversa"},
)
for no in ["rag", "calculadora", "conversa"]:
    builder.add_edge(no, END)
app = builder.compile()

### Slide 10 — draw_mermaid() — o grafo se auto-documenta

In [ ]:
from IPython.display import Image, display

# Renderizar como imagem PNG
try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    # Fallback: imprimir codigo Mermaid (colar em mermaid.live)
    print(app.get_graph().draw_mermaid())

# Saida do draw_mermaid() para o Router desta aula:
# graph TD
#     __start__([start]) --> classificar
#     classificar --> rag
#     classificar --> calculadora
#     classificar --> conversa
#     rag --> __end__([end])
#     calculadora --> __end__([end])
#     conversa --> __end__([end])
# Cole em https://mermaid.live para visualizar

# Inspecionar nos executados com stream()
for evento in app.stream({"mensagens": [HumanMessage("Qual o prazo?")],
                           "rota":"", "iteracoes":0}):
    print(f"No executado: {list(evento.keys())}")
# → No executado: ['classificar']
# → No executado: ['rag']

### Slide 12 — MemorySaver — estado persistente por thread_id

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()

# Compilar COM checkpointer — habilita persistencia e HITL
app = builder.compile(checkpointer=checkpointer)

# thread_id identifica a "sessao" — cada usuario tem seu proprio thread
config_u1 = {"configurable": {"thread_id": "usuario_01"}}
config_u2 = {"configurable": {"thread_id": "usuario_02"}}

# Chamada 1 — usuario 01
app.invoke(
    {"mensagens": [HumanMessage("Qual o prazo de garantia?")],
     "rota":"", "iteracoes":0},
    config=config_u1,
)

# Chamada 2 — mesmo usuario 01 — historico preservado
r = app.invoke(
    {"mensagens": [HumanMessage("E a multa por rescisao?")]},
    config=config_u1,
)
print(len(r["mensagens"]))  # → 4 (2 Human + 2 AI acumulados)

# Inspecionar o estado salvo
snapshot = app.get_state(config_u1)
print(f"Iteracoes: {snapshot.values['iteracoes']}")
print(f"Proximo no: {snapshot.next}")  # [] se terminou, ['nome'] se pausado

### Slide 14 — HITL com interrupt_before — aprovacao antes de acao irreversivel

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# Compilar COM interrupt_before — pausa ANTES do no especificado
app_hitl = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["enviar_email"],  # pausa ANTES deste no
)

config = {"configurable": {"thread_id": "tarefa_01"}}

# FASE 1 — rodar ate a pausa
app_hitl.invoke(
    {"mensagens": [HumanMessage("Gere um rascunho de email")]},
    config=config,
)
# → nos anteriores executaram, PAUSOU antes de "enviar_email"
# → estado salvo no MemorySaver — thread "suspensa"

# FASE 2 — humano le o rascunho
snapshot = app_hitl.get_state(config)
print(f"Rascunho: {snapshot.values['mensagens'][-1].content}")
print(f"No suspenso: {snapshot.next}")  # → ('enviar_email',)

# FASE 3a — humano aprova → continuar (input=None)
aprovado = input("Aprovar envio? (s/n): ")
if aprovado.lower() == "s":
    app_hitl.invoke(None, config=config)  # None = retomar sem novo input
    print("Email enviado!")
# FASE 3b — humano edita → update_state() antes de retomar
else:
    nova_msg = HumanMessage(input("Editar rascunho: "))
    app_hitl.update_state(config, {"mensagens": [nova_msg]})
    app_hitl.invoke(None, config=config)

### Slide 21 — Python novo desta aula

In [ ]:
# 1. TypedDict — dicionario com tipos declarados
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class Estado(TypedDict):
    mensagens: Annotated[list, add_messages]  # reducer: acumula
    contador: int    # sem reducer: substituido no update

# 2. app.get_state(config) — inspecionar estado salvo
snapshot = app.get_state({"configurable":{"thread_id":"abc"}})
snapshot.values    # → dict com estado atual
snapshot.next      # → () se terminou, ('nome_no',) se pausado

# 3. app.update_state(config, values) — editar estado antes de retomar
app.update_state(
    {"configurable":{"thread_id":"abc"}},
    {"mensagens":[HumanMessage("versao editada")]},
)

# 4. app.invoke(None, config) — retomar grafo pausado
app.invoke(None, config=config)
# None = continuar do ponto suspenso sem novo input

# 5. pesquisador.get_graph().draw_mermaid() — auto-documentacao
print(pesquisador.get_graph().draw_mermaid())
# retorna string Mermaid — colar em mermaid.live para visualizar
# ou usar draw_mermaid_png() para renderizar inline no Colab

### Slide 26 — Demo Prática ao Vivo — agente pesquisador com loop condicional

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_community.tools import DuckDuckGoSearchRun
from pydantic import BaseModel
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate

class EstadoPesquisa(TypedDict):
    mensagens: Annotated[list, add_messages]
    resultados_busca: str
    qualidade: float
    iteracoes: int

class Qualidade(BaseModel):
    score: float; suficiente: bool

def node_buscar(state) -> dict:
    bruto = DuckDuckGoSearchRun().run(state["mensagens"][0].content)
    return {"resultados_busca":bruto[:1500], "iteracoes":state["iteracoes"]+1}

def node_avaliar(state) -> dict:
    q = (ChatPromptTemplate.from_template(
        "Pergunta:{p}\nResultado:{r}\nScore suficiencia 0.0-1.0:"
    ) | llm.with_structured_output(Qualidade)).invoke(
        {"p":state["mensagens"][0].content, "r":state["resultados_busca"]}
    )
    return {"qualidade": q.score}

def node_responder(state) -> dict:
    resp = llm.invoke([HumanMessage(
        f"Pergunta:{state['mensagens'][0].content}\nFonte:{state['resultados_busca']}"
    )])
    return {"mensagens":[resp]}

def decidir_continuar(state) -> str:
    # Para se qualidade OK ou atingiu limite de 3 iteracoes
    if state["qualidade"] >= 0.7 or state["iteracoes"] >= 3:
        return "responder"
    return "buscar"  # loop de volta

builder = StateGraph(EstadoPesquisa)
builder.add_node("buscar",node_buscar); builder.add_node("avaliar",node_avaliar)
builder.add_node("responder",node_responder)
builder.add_edge(START,"buscar"); builder.add_edge("buscar","avaliar")
builder.add_conditional_edges("avaliar",decidir_continuar,
                              {"buscar":"buscar","responder":"responder"})
builder.add_edge("responder",END)
pesquisador = builder.compile(checkpointer=MemorySaver())

---

# 💻 Lab do aluno — versão com lacunas

## 📋 Roteiro do Lab

**Lab — Aula 13 · 2º Semestre**  
### Grafo pesquisador com loop condicional para o dominio ★★★

*Grupo 3–4 · 25 minutos · Google Colab*

1. Complete as 5 lacunas — TypedDict com add_messages + tipos; node_buscar com DuckDuckGo; node_responder usando resultados_busca; decidir_continuar com threshold 0.7 e MAX 3; montar o grafo com add_node, add_edge e add_conditional_edges.
2. Visualize o grafo com draw_mermaid() — confirmar que a aresta de volta (avaliar → buscar) aparece no diagrama.
3. Execute 2 perguntas do dominio e observe: quantas iteracoes foram necessarias? O threshold 0.7 foi atingido antes do limite 3 ou precisou do MAX?
4. Experimento com threshold : mudar para 0.5 e 0.9 e documentar a diferenca no numero de iteracoes em celula markdown.

In [ ]:
!pip install langgraph langchain-ollama langchain-community duckduckgo-search -q

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
from pydantic import BaseModel
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_community.tools import DuckDuckGoSearchRun
from IPython.display import Image, display
import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]="https://ollama.com"
os.environ["OLLAMA_API_KEY"]=userdata.get("OLLAMA_API_KEY")
llm=ChatOllama(model="gpt-oss:120b",temperature=0)

# LACUNA 1: Estado com TypedDict
class Estado(TypedDict):
    mensagens: Annotated[list, ___]  # usar add_messages
    resultados_busca: str
    qualidade: ___  # float
    iteracoes: ___  # int

class Qualidade(BaseModel):
    score:float; suficiente:bool

# LACUNA 2: node_buscar
def node_buscar(state:Estado)->dict:
    bruto=DuckDuckGoSearchRun().run(___)  # query = mensagens[0].content
    return {"resultados_busca":bruto[:1500],"iteracoes":state["iteracoes"]+1}

def node_avaliar(state:Estado)->dict:  # PRONTO
    q=(ChatPromptTemplate.from_template("P:{p}\nR:{r}\nScore:")
       |llm.with_structured_output(Qualidade)).invoke(
        {"p":state["mensagens"][0].content,"r":state["resultados_busca"]})
    return {"qualidade":q.score}

# LACUNA 3: node_responder
def node_responder(state:Estado)->dict:
    resp=llm.invoke([HumanMessage(
        f"P:{state['mensagens'][0].content}\nFonte:{state[___]}"
    )])
    return {"mensagens":[resp]}

# LACUNA 4: decidir_continuar (threshold 0.7, MAX 3)
def decidir_continuar(state:Estado)->str:
    if state["qualidade"]>=___ or state["iteracoes"]>=___:
        return ___  # "responder"
    return ___      # "buscar"

# LACUNA 5: montar grafo
builder=StateGraph(Estado)
builder.add_node("buscar",___);builder.add_node("avaliar",___);builder.add_node("responder",___)
builder.add_edge(START,"buscar");builder.add_edge("buscar",___)  # → avaliar
builder.add_conditional_edges("avaliar",___,{"buscar":"buscar","responder":"responder"})
builder.add_edge("responder",END)
pesquisador=builder.compile(checkpointer=MemorySaver())
try: display(Image(pesquisador.get_graph().draw_mermaid_png()))
except: print(pesquisador.get_graph().draw_mermaid())

## 📚 Referências da aula

- Docs LangGraph — Guia completo: StateGraph, checkpointing, HITL. langchain-ai.github.io/langgraph/tutorials/introduction
- Docs LangGraph HITL — interrupt_before, update_state, invoke(None). langchain-ai.github.io/langgraph/concepts/human_in_the_loop
- Blog Anthropic Engineering — "Building Effective Agents" (2025). Secao sobre checkpointing e revisao humana. anthropic.com/engineering/building-effective-agents
- Tool Mermaid Live Editor — Para visualizar o output de draw_mermaid() sem instalar playwright. mermaid.live
- Livro Russell, S.; Norvig, P. — Inteligencia Artificial. 3ª ed. Pearson, 2016. Cap. 3 — Resolucao de problemas como busca: base conceitual dos grafos de estado no LangGraph.
- Livro Bornet, P.; Wirtz, J. et al. — Agentic Artificial Intelligence. World Scientific, 2025. A analogia do "funcionário recém-contratado" e HITL como fase de confiança, não trava permanente — a fundamentação por trás do interrupt_before desta aula.
- Livro Gullí, A. — Agentic Design Patterns. O'Reilly, 2025. Cap. 4 — Reflection: o modelo Producer-Critic que justifica separar os nós buscar e avaliar no grafo pesquisador desta aula.

---

**Proxima Aula — Aula 14 (ultima)** — Spec-Driven Development e Encerramento
  
Retrospectiva do semestre · Spec-Driven Development · LangSmith · Proximos passos na carreira · Encerramento.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*